# Prepare Target-Style SFT Dataset for RAG Alignment

This notebook prepares a small target-style supervised fine-tuning dataset to align the Starlar fine-tuned LLM with the original RAG benchmark answer style.

Goal:
- Use Kaggle QA records with context.
- Match the original RAG benchmark style: short, direct, Turkish answers without source/citation lines.
- Avoid leaking the old 20-question evaluation set into training.
- Save train/validation/test JSONL files for a short alignment fine-tuning run.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import glob
import json
import re
import pandas as pd
import numpy as np

project_path = "/content/drive/MyDrive/turkish_legal_rag"

processed_path = f"{project_path}/data/processed"
outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"
models_path = f"{outputs_path}/models"

kaggle_train_path = f"{processed_path}/kaggle_train_qa.csv"
kaggle_val_path = f"{processed_path}/kaggle_val_qa.csv"
kaggle_test_path = f"{processed_path}/kaggle_test_qa.csv"

old_best_scored_path = f"{metrics_path}/source_aware_article_aware_turkish_bge_rag_testset_scored.csv"

print("Processed path exists:", os.path.exists(processed_path))
print("Metrics path exists:", os.path.exists(metrics_path))

for path in [
    kaggle_train_path,
    kaggle_val_path,
    kaggle_test_path,
    old_best_scored_path
]:
    print(path)
    print("Exists:", os.path.exists(path))
    if os.path.exists(path):
        print("Size MB:", round(os.path.getsize(path) / (1024 * 1024), 2))
    print("-" * 80)

Processed path exists: True
Metrics path exists: True
/content/drive/MyDrive/turkish_legal_rag/data/processed/kaggle_train_qa.csv
Exists: True
Size MB: 65.29
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/kaggle_val_qa.csv
Exists: True
Size MB: 13.91
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/kaggle_test_qa.csv
Exists: True
Size MB: 13.78
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/source_aware_article_aware_turkish_bge_rag_testset_scored.csv
Exists: True
Size MB: 0.04
--------------------------------------------------------------------------------


In [3]:
kaggle_train_df = pd.read_csv(kaggle_train_path)
kaggle_val_df = pd.read_csv(kaggle_val_path)
kaggle_test_df = pd.read_csv(kaggle_test_path)

old_best_df = pd.read_csv(old_best_scored_path)

print("Kaggle train:", kaggle_train_df.shape)
print("Kaggle val:", kaggle_val_df.shape)
print("Kaggle test:", kaggle_test_df.shape)
print("Old best eval:", old_best_df.shape)

print("\nTrain columns:")
print(kaggle_train_df.columns.tolist())

display(kaggle_train_df.head(3))
display(old_best_df[["question", "expected_answer"]].head())

Kaggle train: (9019, 6)
Kaggle val: (1933, 6)
Kaggle test: (1933, 6)
Old best eval: (20, 16)

Train columns:
['soru', 'cevap', 'veri türü', 'kaynak', 'context', 'score']


,soru,cevap,veri türü,kaynak,context,score
0,"Kanunda uygulanabilir bir hüküm yoksa, hâkim n...","Kanunda uygulanabilir bir hüküm yoksa, hâkim ö...",hukuk,Türk Medeni Kanunu,TÜRK MEDENİ KANUNU\r\nKanun Numarası : 4721\r\...,8
1,"Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...","Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...",hukuk,Türkiye Cumhuriyeti Anayasası,ALTINCI KISIM\n\nGEÇİCİ HÜKÜMLER\nGeçici Madde...,10
2,Tapu siciline tescilden önce bir aynî hakkı ka...,"Bir aynî hakkı tescilden önce kazanan kimse, g...",hukuk,Türk Medeni Kanunu,DÖRDÜNCÜ KİTAP\r\nEŞYA HUKUKU\r\nÜÇÜNCÜ KISIM\...,8


,question,expected_answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [4]:
def normalize_kaggle_columns(df):
    df = df.copy()

    rename_map = {}

    for col in df.columns:
        col_lower = str(col).lower().strip()

        if col_lower == "soru":
            rename_map[col] = "question"
        elif col_lower == "cevap":
            rename_map[col] = "answer"
        elif col_lower == "kaynak":
            rename_map[col] = "source"
        elif col_lower == "context":
            rename_map[col] = "context"
        elif col_lower == "score":
            rename_map[col] = "score"
        elif col_lower == "veri türü":
            rename_map[col] = "data_type"

    df = df.rename(columns=rename_map)

    return df


train_df_raw = normalize_kaggle_columns(kaggle_train_df)
val_df_raw = normalize_kaggle_columns(kaggle_val_df)
test_df_raw = normalize_kaggle_columns(kaggle_test_df)

for name, df in [
    ("train", train_df_raw),
    ("val", val_df_raw),
    ("test", test_df_raw)
]:
    print("=" * 80)
    print(name, df.shape)
    print(df.columns.tolist())
    display(df.head(2))

train (9019, 6)
['question', 'answer', 'data_type', 'source', 'context', 'score']


,question,answer,data_type,source,context,score
0,"Kanunda uygulanabilir bir hüküm yoksa, hâkim n...","Kanunda uygulanabilir bir hüküm yoksa, hâkim ö...",hukuk,Türk Medeni Kanunu,TÜRK MEDENİ KANUNU\r\nKanun Numarası : 4721\r\...,8
1,"Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...","Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...",hukuk,Türkiye Cumhuriyeti Anayasası,ALTINCI KISIM\n\nGEÇİCİ HÜKÜMLER\nGeçici Madde...,10


val (1933, 6)
['question', 'answer', 'data_type', 'source', 'context', 'score']


,question,answer,data_type,source,context,score
0,Bayrak Kanunu'nda bayrağın hangi özellikleri b...,"Bayrak Kanunu'nda bayrağın rengi, ölçüleri, şe...",hukuk,Türk Bayrağı Tüzüğü,DÖRDÜNCÜ BÖLÜM\r\nBayrağın Konulabileceği ve Ö...,10
1,Savaşta yalan haber yayma suçunun kapsamı nedir?,"Savaşta yalan haber yayma suçunun kapsamı, sav...",hukuk,Türk Ceza Kanunu,DÖRDÜNCÜ KISIM\r\nMillete ve Devlete Karşı Suç...,9


test (1933, 6)
['question', 'answer', 'data_type', 'source', 'context', 'score']


,question,answer,data_type,source,context,score
0,Suçu ve suçluyu övme suçunun cezasının ne kada...,Suçu ve suçluyu övme suçunun cezasının ne kada...,hukuk,Türk Ceza Kanunu,ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar\r\n\r\nBE...,8
1,"Yayımlatan, bedel ödenmesini isteyebilir mi?","Evet, sözleşmede aksi kararlaştırılmış olmadık...",hukuk,Türk Borçlar Kanunu,İKİNCİ KISIM\r\nÖzel Borç İlişkileri\r\nSEKİZİ...,8


In [5]:
required_cols = ["question", "answer", "context"]

for name, df in [
    ("train", train_df_raw),
    ("val", val_df_raw),
    ("test", test_df_raw)
]:
    missing = [c for c in required_cols if c not in df.columns]
    print(name, "missing:", missing)

    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

train missing: []
val missing: []
test missing: []


In [6]:
def normalize_question_text(text):
    text = str(text).lower()
    text = text.replace("ı", "i")
    text = text.replace("ğ", "g")
    text = text.replace("ü", "u")
    text = text.replace("ş", "s")
    text = text.replace("ö", "o")
    text = text.replace("ç", "c")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


old_eval_questions = set(
    old_best_df["question"].apply(normalize_question_text).tolist()
)

print("Old eval question count:", len(old_eval_questions))

for name, df in [
    ("train", train_df_raw),
    ("val", val_df_raw),
    ("test", test_df_raw)
]:
    overlap_count = df["question"].apply(normalize_question_text).isin(old_eval_questions).sum()
    print(name, "overlap with old eval questions:", overlap_count)

Old eval question count: 20
train overlap with old eval questions: 0
val overlap with old eval questions: 0
test overlap with old eval questions: 0


In [7]:
def clean_text(text):
    text = str(text)
    text = text.replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def prepare_alignment_base_df(df, split_name, min_score=8):
    df = df.copy()

    df["question"] = df["question"].apply(clean_text)
    df["answer"] = df["answer"].apply(clean_text)
    df["context"] = df["context"].apply(clean_text)

    if "source" not in df.columns:
        df["source"] = "unknown"

    if "score" not in df.columns:
        df["score"] = np.nan

    # score varsa yüksek kaliteli olanları al
    if df["score"].notna().sum() > 0:
        df = df[df["score"] >= min_score].copy()

    # Eski 20 eval sorusuyla birebir overlap varsa çıkar
    df["normalized_question"] = df["question"].apply(normalize_question_text)
    df = df[~df["normalized_question"].isin(old_eval_questions)].copy()

    # Basit kalite filtreleri
    df = df.dropna(subset=["question", "answer", "context"]).copy()

    df = df[
        (df["question"].str.len() >= 8) &
        (df["answer"].str.len() >= 5) &
        (df["context"].str.len() >= 50)
    ].copy()

    # Aşırı uzun cevapları çıkaralım; eski benchmark kısa/direkt cevap istiyor
    df = df[df["answer"].str.len() <= 1200].copy()

    # Context çok uzun olabilir; sonra prompt sırasında kırpacağız
    df["split_source"] = split_name

    return df.reset_index(drop=True)


clean_train_base_df = prepare_alignment_base_df(train_df_raw, "kaggle_train", min_score=8)
clean_val_base_df = prepare_alignment_base_df(val_df_raw, "kaggle_val", min_score=8)
clean_test_base_df = prepare_alignment_base_df(test_df_raw, "kaggle_test", min_score=8)

print("Clean train:", clean_train_base_df.shape)
print("Clean val:", clean_val_base_df.shape)
print("Clean test:", clean_test_base_df.shape)

display(clean_train_base_df[["question", "answer", "source", "score", "context"]].head(3))

Clean train: (8039, 8)
Clean val: (1704, 8)
Clean test: (1739, 8)


,question,answer,source,score,context
0,"Kanunda uygulanabilir bir hüküm yoksa, hâkim n...","Kanunda uygulanabilir bir hüküm yoksa, hâkim ö...",Türk Medeni Kanunu,8,TÜRK MEDENİ KANUNU\n\nKanun Numarası : 4721\n\...
1,"Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...","Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...",Türkiye Cumhuriyeti Anayasası,10,ALTINCI KISIM\n\nGEÇİCİ HÜKÜMLER\nGeçici Madde...
2,Tapu siciline tescilden önce bir aynî hakkı ka...,"Bir aynî hakkı tescilden önce kazanan kimse, g...",Türk Medeni Kanunu,8,DÖRDÜNCÜ KİTAP\n\nEŞYA HUKUKU\n\nÜÇÜNCÜ KISIM\...


In [17]:
TURKISH_STOPWORDS = {
    "ve", "veya", "ile", "için", "bu", "şu", "o", "bir", "de", "da", "mi", "mı", "mu", "mü",
    "ne", "nedir", "nasıl", "hangi", "olarak", "göre", "olan", "olduğu", "olur", "ise",
    "daha", "çok", "az", "en", "gibi", "kadar", "tarafından", "hakkında", "şekilde",
    "kanun", "madde", "hukuk", "ilgili"
}

def normalize_for_overlap(text):
    text = str(text).lower()
    text = text.replace("ı", "i")
    text = text.replace("ğ", "g")
    text = text.replace("ü", "u")
    text = text.replace("ş", "s")
    text = text.replace("ö", "o")
    text = text.replace("ç", "c")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def content_tokens(text):
    tokens = normalize_for_overlap(text).split()
    tokens = [
        t for t in tokens
        if len(t) >= 4 and t not in TURKISH_STOPWORDS
    ]
    return tokens

def answer_context_recall(answer, context):
    answer_tokens = content_tokens(answer)
    context_tokens = set(content_tokens(context))

    if len(answer_tokens) == 0:
        return 0.0

    matched = sum(1 for t in answer_tokens if t in context_tokens)
    return matched / len(answer_tokens)

def add_grounding_scores(df):
    df = df.copy()

    df["answer_context_recall"] = df.apply(
        lambda row: answer_context_recall(row["answer"], row["context"]),
        axis=1
    )

    return df

clean_train_base_df = add_grounding_scores(clean_train_base_df)
clean_val_base_df = add_grounding_scores(clean_val_base_df)
clean_test_base_df = add_grounding_scores(clean_test_base_df)

for name, df in [
    ("train", clean_train_base_df),
    ("val", clean_val_base_df),
    ("test", clean_test_base_df)
]:
    print("=" * 80)
    print(name)
    display(df["answer_context_recall"].describe())
    print("Low grounding < 0.35:", (df["answer_context_recall"] < 0.35).sum())
    print("Medium/high grounding >= 0.35:", (df["answer_context_recall"] >= 0.35).sum())

train


,answer_context_recall
count,8039.000000
mean,0.583011
std,0.244083
min,0.000000
25%,0.400000
50%,0.578947
75%,0.761905
max,1.000000


Low grounding < 0.35: 1512
Medium/high grounding >= 0.35: 6527
val


,answer_context_recall
count,1704.000000
mean,0.588218
std,0.245551
min,0.000000
25%,0.412465
50%,0.583333
75%,0.767889
max,1.000000


Low grounding < 0.35: 306
Medium/high grounding >= 0.35: 1398
test


,answer_context_recall
count,1739.000000
mean,0.572523
std,0.246674
min,0.025000
25%,0.382979
50%,0.571429
75%,0.750000
max,1.000000


Low grounding < 0.35: 358
Medium/high grounding >= 0.35: 1381


In [18]:
GROUNDING_THRESHOLD = 0.35

before_counts = {
    "train": len(clean_train_base_df),
    "val": len(clean_val_base_df),
    "test": len(clean_test_base_df)
}

clean_train_base_df = clean_train_base_df[
    clean_train_base_df["answer_context_recall"] >= GROUNDING_THRESHOLD
].copy().reset_index(drop=True)

clean_val_base_df = clean_val_base_df[
    clean_val_base_df["answer_context_recall"] >= GROUNDING_THRESHOLD
].copy().reset_index(drop=True)

clean_test_base_df = clean_test_base_df[
    clean_test_base_df["answer_context_recall"] >= GROUNDING_THRESHOLD
].copy().reset_index(drop=True)

after_counts = {
    "train": len(clean_train_base_df),
    "val": len(clean_val_base_df),
    "test": len(clean_test_base_df)
}

print("Before:", before_counts)
print("After:", after_counts)

display(clean_train_base_df[[
    "question",
    "answer",
    "answer_context_recall",
    "context"
]].head(5))

Before: {'train': 8039, 'val': 1704, 'test': 1739}
After: {'train': 6527, 'val': 1398, 'test': 1381}


,question,answer,answer_context_recall,context
0,"Kanunda uygulanabilir bir hüküm yoksa, hâkim n...","Kanunda uygulanabilir bir hüküm yoksa, hâkim ö...",0.613636,TÜRK MEDENİ KANUNU\n\nKanun Numarası : 4721\n\...
1,"Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...","Yargıtay Genel Kurulu, Yargıtay üyeleri arasın...",0.818182,ALTINCI KISIM\n\nGEÇİCİ HÜKÜMLER\nGeçici Madde...
2,Tapu siciline tescilden önce bir aynî hakkı ka...,"Bir aynî hakkı tescilden önce kazanan kimse, g...",1.000000,DÖRDÜNCÜ KİTAP\n\nEŞYA HUKUKU\n\nÜÇÜNCÜ KISIM\...
3,Eskimiş bayrakların yok edilmesi için hangi ba...,"İçişleri Bakanlığı, Milli Savunma Bakanlığı, D...",0.772727,YEDİNCİ BÖLÜM\n\nTescil ve Müsaade İşlemleri\n...
4,Göçmen kaçakçılığı suçunun tüzel kişiler açısı...,Göçmen kaçakçılığı suçunun tüzel kişiler açısı...,0.500000,BİRİNCİ KISIM\n\nUluslararası Suçlar\n\nİKİNCİ...


In [19]:
for name, df in [
    ("clean_train", clean_train_base_df),
    ("clean_val", clean_val_base_df),
    ("clean_test", clean_test_base_df)
]:
    print("=" * 80)
    print(name)
    df["question_len"] = df["question"].str.len()
    df["answer_len"] = df["answer"].str.len()
    df["context_len"] = df["context"].str.len()

    display(df[["question_len", "answer_len", "context_len"]].describe())

    print("Very long context > 5000:", (df["context_len"] > 5000).sum())
    print("Very long answer > 800:", (df["answer_len"] > 800).sum())

clean_train


,question_len,answer_len,context_len
count,6527.000000,6527.000000,6527.000000
mean,83.326490,317.918186,7097.257699
std,34.483429,153.681003,5189.640421
min,13.000000,15.000000,553.000000
25%,59.000000,208.000000,3411.000000
50%,77.000000,297.000000,6040.000000
75%,101.000000,402.000000,9183.000000
max,338.000000,1199.000000,34597.000000


Very long context > 5000: 3784
Very long answer > 800: 63
clean_val


,question_len,answer_len,context_len
count,1398.000000,1398.000000,1398.00000
mean,83.795422,309.706009,7002.11588
std,35.465707,150.101279,5302.62379
min,13.000000,15.000000,553.00000
25%,58.000000,203.000000,3290.00000
50%,78.000000,287.000000,5948.50000
75%,102.000000,398.000000,8951.00000
max,304.000000,1121.000000,34597.00000


Very long context > 5000: 777
Very long answer > 800: 10
clean_test


,question_len,answer_len,context_len
count,1381.000000,1381.000000,1381.000000
mean,83.984794,322.264301,7095.464881
std,34.727226,159.057351,5148.970750
min,16.000000,32.000000,800.000000
25%,60.000000,209.000000,3411.000000
50%,79.000000,301.000000,6160.000000
75%,101.000000,398.000000,8926.000000
max,370.000000,1171.000000,34597.000000


Very long context > 5000: 819
Very long answer > 800: 20


In [20]:
TARGET_SYSTEM_INSTRUCTION = """Sen Türk hukuk soruları için çalışan dikkatli bir RAG asistanısın.
Cevabı yalnızca verilen bağlama dayanarak üret.
Bağlamda açıkça bulunmayan bilgileri uydurma.
Cevap kısa, doğrudan ve Türkçe olmalı.
Kaynak, citation, chunk id veya bağlam numarası yazma.
Sadece nihai cevabı yaz."""


def truncate_context_for_alignment(context, max_chars=2500):
    context = str(context)

    if len(context) <= max_chars:
        return context

    return context[:max_chars].strip()


def build_target_style_sft_text(question, answer, context):
    question = clean_text(question)
    answer = clean_text(answer)
    context = truncate_context_for_alignment(context, max_chars=2500)

    return f"""<s>[INST] {TARGET_SYSTEM_INSTRUCTION}

Bağlam:
{context}

Soru:
{question}

Cevap: [/INST] {answer}</s>"""

In [21]:
def add_sft_text(df):
    df = df.copy()

    df["text"] = df.apply(
        lambda row: build_target_style_sft_text(
            question=row["question"],
            answer=row["answer"],
            context=row["context"]
        ),
        axis=1
    )

    df["text_len"] = df["text"].str.len()

    return df


alignment_train_full_df = add_sft_text(clean_train_base_df)
alignment_val_full_df = add_sft_text(clean_val_base_df)
alignment_test_full_df = add_sft_text(clean_test_base_df)

print("Train text sample:")
print(alignment_train_full_df.iloc[0]["text"][:2500])

print("\nShapes:")
print("train:", alignment_train_full_df.shape)
print("val:", alignment_val_full_df.shape)
print("test:", alignment_test_full_df.shape)

Train text sample:
<s>[INST] Sen Türk hukuk soruları için çalışan dikkatli bir RAG asistanısın.
Cevabı yalnızca verilen bağlama dayanarak üret.
Bağlamda açıkça bulunmayan bilgileri uydurma.
Cevap kısa, doğrudan ve Türkçe olmalı.
Kaynak, citation, chunk id veya bağlam numarası yazma.
Sadece nihai cevabı yaz.

Bağlam:
TÜRK MEDENİ KANUNU

Kanun Numarası : 4721

Kabul Tarihi : 22/11/2001

Yayımlandığı Resmî Gazete : Tarihi: 8/12/2001 Sayı: 24607

Yayımlandığı Düstur : Tertip: 5 Cilt: 41

Durumu: 3/12/2001 tarih ve 4722 sayılı “Türk Medenî Kanununun Yürürlüğü ve Uygulama

Şekli Hakkında Kanun”un 22 nci Maddesi uyarınca; yeni düzenlemeler yapılıncaya kadar, 

yürürlükteki tüzük ve yönetmeliklerin Türk Medenî Kanunu’na aykırı olmayan hükümlerinin 

uygulanmasına devam edileceğinden, gerektiğinde “Tüzükler Külliyatı” ile “Yönetmelikler 

Külliyatı”nın kanunlara göre (743 sayılı Kanuna göre) düzenlenen nümerik fihriste, 4721 sayılı 

Kanuna dayanılarak yürürlüğe konulan tüzük için ise 4721 sayı

In [22]:
for name, df in [
    ("alignment_train_full", alignment_train_full_df),
    ("alignment_val_full", alignment_val_full_df),
    ("alignment_test_full", alignment_test_full_df)
]:
    print("=" * 80)
    print(name)
    display(df[["text_len", "question_len", "answer_len", "context_len"]].describe())
    print("Texts > 5000 chars:", (df["text_len"] > 5000).sum())

alignment_train_full


,text_len,question_len,answer_len,context_len
count,6527.000000,6527.000000,6527.000000,6527.000000
mean,3144.253869,83.326490,317.918186,7097.257699
std,318.387821,34.483429,153.681003,5189.640421
min,1020.000000,13.000000,15.000000,553.000000
25%,3075.000000,59.000000,208.000000,3411.000000
50%,3184.000000,77.000000,297.000000,6040.000000
75%,3303.000000,101.000000,402.000000,9183.000000
max,4127.000000,338.000000,1199.000000,34597.000000


Texts > 5000 chars: 0
alignment_val_full


,text_len,question_len,answer_len,context_len
count,1398.000000,1398.000000,1398.000000,1398.00000
mean,3127.831903,83.795422,309.706009,7002.11588
std,337.694098,35.465707,150.101279,5302.62379
min,1017.000000,13.000000,15.000000,553.00000
25%,3063.250000,58.000000,203.000000,3290.00000
50%,3175.000000,78.000000,287.000000,5948.50000
75%,3295.000000,102.000000,398.000000,8951.00000
max,3982.000000,304.000000,1121.000000,34597.00000


Texts > 5000 chars: 0
alignment_test_full


,text_len,question_len,answer_len,context_len
count,1381.000000,1381.000000,1381.000000,1381.000000
mean,3154.271542,83.984794,322.264301,7095.464881
std,313.637977,34.727226,159.057351,5148.970750
min,1434.000000,16.000000,32.000000,800.000000
25%,3081.000000,60.000000,209.000000,3411.000000
50%,3197.000000,79.000000,301.000000,6160.000000
75%,3307.000000,101.000000,398.000000,8926.000000
max,4121.000000,370.000000,1171.000000,34597.000000


Texts > 5000 chars: 0


In [23]:
TRAIN_SAMPLE_SIZE = min(len(alignment_train_full_df), 1500)
VAL_SAMPLE_SIZE = min(len(alignment_val_full_df), 200)
TEST_SAMPLE_SIZE = min(len(alignment_test_full_df), 200)

alignment_train_df = alignment_train_full_df.sample(
    n=TRAIN_SAMPLE_SIZE,
    random_state=42
).reset_index(drop=True)

alignment_val_df = alignment_val_full_df.sample(
    n=VAL_SAMPLE_SIZE,
    random_state=42
).reset_index(drop=True)

alignment_test_df = alignment_test_full_df.sample(
    n=TEST_SAMPLE_SIZE,
    random_state=42
).reset_index(drop=True)

print("Alignment train:", alignment_train_df.shape)
print("Alignment val:", alignment_val_df.shape)
print("Alignment test:", alignment_test_df.shape)

display(alignment_train_df[["question", "answer", "source", "score", "text"]].head(3))

Alignment train: (1500, 14)
Alignment val: (200, 14)
Alignment test: (200, 14)


,question,answer,source,score,text
0,Kamu davasının açılmasının ertelenmesi süresin...,Erteleme süresi içinde şüpheli kasıtlı bir suç...,Ceza Muhakemesi Kanunu,8,<s>[INST] Sen Türk hukuk soruları için çalışan...
1,Eğer görevlendirilen avukat yargılama sırasınd...,Eğer görevlendirilen avukat yargılama sırasınd...,Ceza Muhakemesi Kanunu,8,<s>[INST] Sen Türk hukuk soruları için çalışan...
2,"Kiraya veren, yasal hükümlere uymazsa eski kir...","Kiraya veren, yasal hükümlere uymazsa, eski ki...",Türk Borçlar Kanunu,8,<s>[INST] Sen Türk hukuk soruları için çalışan...


In [24]:
for name, df in [
    ("alignment_train", alignment_train_df),
    ("alignment_val", alignment_val_df),
    ("alignment_test", alignment_test_df)
]:
    overlap_count = df["question"].apply(normalize_question_text).isin(old_eval_questions).sum()
    print(name, "overlap with old 20 eval questions:", overlap_count)

    if overlap_count > 0:
        raise ValueError(f"Leakage detected in {name}: {overlap_count} questions overlap with old eval set.")

alignment_train overlap with old 20 eval questions: 0
alignment_val overlap with old 20 eval questions: 0
alignment_test overlap with old 20 eval questions: 0


In [25]:
target_alignment_train_path = f"{processed_path}/target_style_alignment_sft_train.jsonl"
target_alignment_val_path = f"{processed_path}/target_style_alignment_sft_val.jsonl"
target_alignment_test_path = f"{processed_path}/target_style_alignment_sft_test.jsonl"

columns_to_save = [
    "text",
    "question",
    "answer",
    "context",
    "source",
    "score",
    "answer_context_recall",
    "split_source"
]

alignment_train_df[columns_to_save].to_json(
    target_alignment_train_path,
    orient="records",
    lines=True,
    force_ascii=False
)

alignment_val_df[columns_to_save].to_json(
    target_alignment_val_path,
    orient="records",
    lines=True,
    force_ascii=False
)

alignment_test_df[columns_to_save].to_json(
    target_alignment_test_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved:")
print(target_alignment_train_path)
print(target_alignment_val_path)
print(target_alignment_test_path)

Saved:
/content/drive/MyDrive/turkish_legal_rag/data/processed/target_style_alignment_sft_train.jsonl
/content/drive/MyDrive/turkish_legal_rag/data/processed/target_style_alignment_sft_val.jsonl
/content/drive/MyDrive/turkish_legal_rag/data/processed/target_style_alignment_sft_test.jsonl


In [26]:
alignment_data_summary_df = pd.DataFrame([{
    "dataset_name": "target_style_alignment_sft",
    "source_train_file": kaggle_train_path,
    "source_val_file": kaggle_val_path,
    "source_test_file": kaggle_test_path,
    "min_score": 8,
    "old_eval_questions_removed": True,
    "target_style": "short_direct_answer_without_citation",
    "grounding_threshold": GROUNDING_THRESHOLD,
    "train_mean_answer_context_recall": alignment_train_df["answer_context_recall"].mean(),
    "val_mean_answer_context_recall": alignment_val_df["answer_context_recall"].mean(),
    "test_mean_answer_context_recall": alignment_test_df["answer_context_recall"].mean(),
    "train_rows": len(alignment_train_df),
    "val_rows": len(alignment_val_df),
    "test_rows": len(alignment_test_df),
    "max_context_chars": 2500,
    "train_path": target_alignment_train_path,
    "val_path": target_alignment_val_path,
    "test_path": target_alignment_test_path
}])

alignment_summary_path = f"{metrics_path}/target_style_alignment_sft_data_preparation_summary.csv"

alignment_data_summary_df.to_csv(
    alignment_summary_path,
    index=False,
    encoding="utf-8-sig"
)

display(alignment_data_summary_df)

print("Summary saved:", alignment_summary_path)

,dataset_name,source_train_file,source_val_file,source_test_file,min_score,old_eval_questions_removed,target_style,grounding_threshold,train_mean_answer_context_recall,val_mean_answer_context_recall,test_mean_answer_context_recall,train_rows,val_rows,test_rows,max_context_chars,train_path,val_path,test_path
0,target_style_alignment_sft,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...,8,True,short_direct_answer_without_citation,0.35,0.654759,0.661045,0.66603,1500,200,200,2500,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...


Summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/target_style_alignment_sft_data_preparation_summary.csv


In [27]:
for i, row in alignment_train_df.head(5).iterrows():
    print("=" * 120)
    print("QUESTION:")
    print(row["question"])

    print("\nANSWER:")
    print(row["answer"])

    print("\nCONTEXT PREVIEW:")
    print(row["context"][:800])

    print("\nSFT TEXT PREVIEW:")
    print(row["text"][:1500])

QUESTION:
Kamu davasının açılmasının ertelenmesi süresince şüpheli kasıtlı bir suç işlerse ne olur?

ANSWER:
Erteleme süresi içinde şüpheli kasıtlı bir suç işlerse kamu davası açılır. Bu, erteleme kararının kaldırıldığı ve şüphelinin suçundan dolayı yargılanacağı anlamına gelir.

CONTEXT PREVIEW:
İKİNCİ KİTAP

Soruşturma

İKİNCİ KISIM

Kamu Davasının Açılması

 

BİRİNCİ BÖLÜM

Kamu Davasının Açılması

Kamu davasını açma görevi

Madde 170 – (1) Kamu davasını açma görevi, Cumhuriyet savcısı tarafından yerine getirilir.

(2) Soruşturma evresi sonunda toplanan deliller, suçun işlendiği hususunda yeterli şüphe oluşturuyorsa; Cumhuriyet savcısı, bir iddianame düzenler.

(3) Görevli ve yetkili mahkemeye hitaben düzenlenen iddianamede;

a) Şüphelinin kimliği,

b) Müdafii,

c) Maktul, mağdur veya suçtan zarar görenin kimliği,

d) Mağdurun veya suçtan zarar görenin vekili veya kanunî temsilcisi,

e) Açıklanmasında sakınca bulunmaması halinde ihbarda bulunan kişinin kimliği,

f) Şikâyette buluna

In [28]:
files_to_check = [
    target_alignment_train_path,
    target_alignment_val_path,
    target_alignment_test_path,
    alignment_summary_path
]

print("FINAL CHECK")
print("=" * 80)

for file in files_to_check:
    print(file)
    print("Exists:", os.path.exists(file))
    if os.path.exists(file):
        print("Size MB:", round(os.path.getsize(file) / (1024 * 1024), 4))
    print("-" * 80)

print("Notebook 23 completed.")

FINAL CHECK
/content/drive/MyDrive/turkish_legal_rag/data/processed/target_style_alignment_sft_train.jsonl
Exists: True
Size MB: 17.232
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/target_style_alignment_sft_val.jsonl
Exists: True
Size MB: 2.3128
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/target_style_alignment_sft_test.jsonl
Exists: True
Size MB: 2.4245
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/target_style_alignment_sft_data_preparation_summary.csv
Exists: True
Size MB: 0.0009
--------------------------------------------------------------------------------
Notebook 23 completed.
